# CUSTOMER CHURN & REVENUE INTELLIGENCE
## Part 3: Predictive Machine Learning Pipeline & Revenue at Risk Engine
**Author:** Senior Data Analyst & Machine Learning Practitioner  
**Algorithms:** Logistic Regression & Random Forest Classifier  
**Evaluation:** Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix  
**Business Goal:** Predict customer churn probabilities before cancellation happens and quantify exact dollars at risk to enable targeted intervention.

---
### Table of Contents:
1. Formulation & Preventing Data Leakage
2. Feature Encoding & Stratified Split
3. Model Training: Logistic Regression (Benchmark)
4. Model Training: Random Forest Classifier (Ensemble)
5. Model Evaluation & Comparison (ROC-AUC & Classification Report)
6. Confusion Matrix & Cost of False Negatives
7. Feature Importance & Explainability
8. Probability Scoring & Categorization into Risk Tiers
9. Revenue-at-Risk Quantification
10. Exporting High-Priority Intervention List


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../dataset/customer_churn_cleaned.csv')
print(f"Loaded {len(df):,} customer records for predictive modeling.")


## 1. Feature Engineering & Strict Data Leakage Prevention
To prevent data leakage:
- Target variable: `Is_Churn` derived from `Churn_Status`.
- Exclude future-looking post-churn variables: `Churn_Reason` and `Last_Login_Date` (which correlate perfectly with exit timing).
- Categorical features are encoded numerically.
- Scaling parameters are fitted **strictly on training data** and transformed on holdout test data.

In [ ]:
ml_df = df.copy()

# Target
ml_df['Is_Churn'] = (ml_df['Churn_Status'] == 'Yes').astype(int)

# One-Hot / Binary encodings
ml_df['Contract_MonthToMonth'] = (ml_df['Contract_Type'] == 'Month-to-Month').astype(int)
ml_df['Contract_TwoYear'] = (ml_df['Contract_Type'] == 'Two Year').astype(int)
ml_df['Payment_ElectronicCheck'] = (ml_df['Payment_Method'] == 'Electronic Check').astype(int)
ml_df['Payment_CreditCardAuto'] = (ml_df['Payment_Method'] == 'Credit Card (Automatic)').astype(int)
ml_df['Internet_FiberOptic'] = (ml_df['Internet_Service'] == 'Fiber Optic').astype(int)
ml_df['Internet_None'] = (ml_df['Internet_Service'] == 'None').astype(int)
ml_df['Discount_Yes'] = (ml_df['Discount_Applied'] == 'Yes').astype(int)
ml_df['Gender_Male'] = (ml_df['Gender'] == 'Male').astype(int)

feature_cols = [
    'Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges',
    'Support_Tickets', 'Usage_Hours', 'Contract_MonthToMonth',
    'Contract_TwoYear', 'Payment_ElectronicCheck', 'Payment_CreditCardAuto',
    'Internet_FiberOptic', 'Internet_None', 'Discount_Yes', 'Gender_Male'
]

X = ml_df[feature_cols]
y = ml_df['Is_Churn']

print(f"Features: {len(feature_cols)} predictor variables selected.")
print(f"Target distribution: {y.value_counts(normalize=True).round(3).to_dict()}")


## 2. Stratified Train / Test Partitioning

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]:,} records")
print(f"Holdout testing set: {X_test.shape[0]:,} records")


## 3. Model Training & Evaluation
We fit a regularized Logistic Regression and a 200-tree Random Forest Classifier.

In [ ]:
# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=9, min_samples_split=10, random_state=42, n_jobs=1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")
print(f"Random Forest ROC-AUC:       {rf_auc:.4f}")


### Detailed Classification Reports

In [ ]:
print("=== LOGISTIC REGRESSION CLASSIFICATION REPORT ===")
print(classification_report(y_test, lr_preds, target_names=['Active', 'Churned']))

print("
=== RANDOM FOREST CLASSIFICATION REPORT ===")
print(classification_report(y_test, rf_preds, target_names=['Active', 'Churned']))


## 4. Confusion Matrix Analysis

In [ ]:
cm = confusion_matrix(y_test, rf_preds)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Retained', 'Actual Churned'],
    columns=['Predicted Retained', 'Predicted Churned']
)
print("Random Forest Confusion Matrix:")
cm_df


## 5. Feature Importance & Key Risk Drivers

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance_df['Importance_%'] = (feature_importance_df['Importance'] * 100).round(2)
feature_importance_df


## 6. Full Customer Probability Scoring & Revenue at Risk
We apply the trained model to all active customers to calculate their individual churn probability and quantify contracted revenue exposure.

In [ ]:
# Score full database
all_scaled = scaler.transform(X)
df['Predicted_Churn_Prob'] = rf_model.predict_proba(X)[:, 1].round(4)

# Create probability risk tiers
df['ML_Risk_Tier'] = pd.cut(
    df['Predicted_Churn_Prob'],
    bins=[-0.01, 0.25, 0.50, 0.75, 1.01],
    labels=['Low Risk (<25%)', 'Moderate Risk (25-50%)', 'High Risk (50-75%)', 'Critical Risk (>75%)']
)

# Active Customer Risk Summary
active_customers_df = df[df['Churn_Status'] == 'No'].copy()
risk_revenue_summary = active_customers_df.groupby('ML_Risk_Tier', observed=False).agg(
    Customer_Count=('CustomerID', 'count'),
    Monthly_Revenue_At_Risk=('Monthly_Charges', 'sum'),
    Avg_Monthly_Charges=('Monthly_Charges', 'mean'),
    Avg_Support_Tickets=('Support_Tickets', 'mean'),
    Avg_Tenure=('Tenure_Months', 'mean')
).reset_index()

risk_revenue_summary['Annualized_Revenue_At_Risk'] = risk_revenue_summary['Monthly_Revenue_At_Risk'] * 12
risk_revenue_summary['%_of_Active_MRR'] = (
    risk_revenue_summary['Monthly_Revenue_At_Risk'] / active_customers_df['Monthly_Charges'].sum() * 100
).round(2)

risk_revenue_summary


## 7. High-Priority Accounts for Immediate Retention Campaign
Exporting active accounts with >50% predicted churn risk and Monthly Charges > $75.

In [ ]:
intervention_list = active_customers_df[
    (active_customers_df['Predicted_Churn_Prob'] >= 0.50)
].sort_values(by='Monthly_Charges', ascending=False)[
    ['CustomerID', 'City', 'Subscription_Plan', 'Contract_Type',
     'Monthly_Charges', 'Tenure_Months', 'Support_Tickets', 'Usage_Hours',
     'Predicted_Churn_Prob', 'ML_Risk_Tier']
]

print(f"Total High/Critical Risk Accounts: {len(intervention_list):,}")
print(f"Total Monthly MRR at Immediate Risk: ${intervention_list['Monthly_Charges'].sum():,.2f}")
intervention_list.head(10)
